# 82514 · Sesión S17 — Percepción aprendida: núcleos aprendidos, entrenar y probar, y un detector medido

**Bloque 6** · lunes 16 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** De los núcleos diseñados a los aprendidos: un núcleo de 3 × 3 ajustado por mínimos cuadrados que se parece a Sobel; un clasificador de una neurona, «¿pieza o mesa?», entrenado y probado con pocos y con muchos ejemplos y con otra luz (el sobreajuste y el dominio); el detector clásico del cuaderno S17a, con su umbral elegido en 20 escenas con verdad conocida y comprobado en 40 nuevas (IoU, precisión y exhaustividad); y un detector aprendido de verdad, NanoDet-Plus, ejecutado en la CPU con OpenCV: sus cajas, su latencia y sus primeros núcleos.

**Se apoya en:** Corke (2023), cap. 12; LeCun et al. (1998); Redmon et al. (2016); Kirillov et al. (2023); documentación de OpenCV (módulo dnn) y de scikit-learn; NanoDet-Plus, de OpenCV Zoo (Apache 2.0).

**Cómo usarlo en clase.** Sigue el guion de la sesión S17 en las diapositivas del bloque 6. Ejecuta la celda de instalación una sola vez al empezar; en Colab tarda unos veinte segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos veinte segundos.
import subprocess, sys, importlib.metadata as md

OPENCV = '4.14.0.94'          # la versión de todo el bloque 6 (cv2.__version__ = '4.14.0')
FIJAS = {'scikit-learn': '1.6.1', 'scikit-image': '0.25.2'}               # las de Colab: se comprueban y, si cambian, se instalan antes de importarlas

def opencv_activo():
    """Versión de OpenCV que se importaría, mirada en otro proceso: Colab trae tres paquetes de OpenCV que
    comparten la carpeta cv2, y lo que dice pip no basta para saber cuál se carga."""
    r = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'], capture_output=True, text=True)
    return r.stdout.strip()

def fijar():
    if opencv_activo() != OPENCV.rsplit('.', 1)[0]:
        print('Instalando OpenCV', OPENCV, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-python-headless=={OPENCV}'], check=False)
    for paquete, version in FIJAS.items():
        try:
            ya = md.version(paquete)
        except md.PackageNotFoundError:
            ya = None
        if ya != version:
            print('Instalando', paquete, version, '...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={version}'], check=False)

fijar()
import os, json, time, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import cv2
warnings.filterwarnings('ignore')
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def sin_ejes(*axs):
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])

print('OpenCV', cv2.__version__, '· numpy', np.__version__, '· en Colab:', EN_COLAB)
if cv2.__version__ != OPENCV.rsplit('.', 1)[0]:
    print('AVISO: este cuaderno se probó con OpenCV', OPENCV, '. Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar esta celda.')
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

# El detector: NanoDet-Plus (OpenCV Zoo, Apache 2.0; entrenado con COCO, 80 clases), 3,8 MB, de una revisión fijada.
NANODET = ('https://huggingface.co/opencv/object_detection_nanodet/resolve/5bfd47077350a726ad440dd7bd1e1e35e8ebcfb2/'
           'object_detection_nanodet_2022nov.onnx')
COPIA = ('https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/object_detection_nanodet/'
         'object_detection_nanodet_2022nov.onnx')
HUELLA = '4b82da9944b88577175ee23a459dce2e26e6e4be573def65b1055dc2d9720186'   # SHA-256 del fichero

def descargar_detector():
    import hashlib, urllib.request
    ruta = os.path.join('.' if EN_COLAB else tempfile.gettempdir(), 'nanodet_2022nov.onnx')
    for k, url in enumerate((NANODET, COPIA), 1):
        try:
            if not (os.path.exists(ruta) and hashlib.sha256(open(ruta, 'rb').read()).hexdigest() == HUELLA):
                datos = urllib.request.urlopen(url, timeout=20).read()
                if hashlib.sha256(datos).hexdigest() != HUELLA:
                    continue
                open(ruta, 'wb').write(datos)
            return ruta
        except Exception as e:
            print(f'No se pudo descargar de la copia {k} ({type(e).__name__}).')
    print('AVISO: sin el detector. La sección 4 imprime lo que habría salido (medido con OpenCV 4.14.0).')
    print('  La copia 2 apunta a la rama principal de OpenCV Zoo: si cambia el fichero, su huella no cuadra y no se usa.')
    return None

RUTA_DETECTOR = descargar_detector()
import sklearn, skimage
print('scikit-learn', sklearn.__version__, '· scikit-image', skimage.__version__, '· detector:', 'listo' if RUTA_DETECTOR else 'no')

> **Aviso para Colab.** Ábrelo al acabar el taller S17a y ejecuta la primera celda: fija OpenCV 4.14.0 y descarga el detector NanoDet-Plus (3,8 MB), en unos veinte segundos la primera vez. Así, a la vuelta del descanso, está listo.
>
> **En clase (ocho minutos):** el simulador y las secciones 1 y 2, sin detenerte (sus cifras ya están en las diapositivas); la sección 3, con el ejercicio de clase, que se pone en común al acabar; y la sección 4.
>
> **En casa:** los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 0. La mesa del puesto de picking

*Ejecútalo sin detenerte.* Es el mismo simulador del cuaderno S17a: la cámara del curso (f = 800 px, k₁ = −0,25, k₂ = 0,06), cenital a 1 m de la mesa, y piezas de 10 mm de alto, con la luz del techo o con una lámpara a la derecha. Se conoce la verdad píxel a píxel: qué píxeles son la cara de arriba de cada pieza.

In [ ]:
# @title El simulador de la mesa (ejecutadla sin abrirla)
# El simulador de la mesa del puesto de picking. Ejecútalo sin detenerte: es la cámara de S16 apuntando a la mesa.
# Cada píxel (submuestreado 2 x 2) es un rayo que sale de la cámara con su K y su distorsión, se gira al mundo con la
# pose de la cámara y se corta con la mesa y con las piezas. Después, desenfoque, ruido y 8 bits, como una foto.
W, H = 640, 480
F = 800.0
K_CURSO = np.array([[F, 0, W/2], [0, F, H/2], [0, 0, 1.0]])
DIST_CURSO = np.array([-0.25, 0.06, 0.0, 0.0, 0.0])
A1, A2 = 0.35, 0.20
TC = np.array([0.30, 0.20, 1.0])
R_0C = np.diag([1.0, -1.0, -1.0])            # Rx(180°): la camara mira hacia abajo
P_PIEZA = np.array([0.40, 0.20])
H_PIEZA, D_PIEZA = 0.010, 0.040

T_0C_NOMINAL = np.eye(4); T_0C_NOMINAL[:3, :3] = R_0C; T_0C_NOMINAL[:3, 3] = TC   # el plano del montaje


def T_C0_verdad():
    M = np.eye(4); M[:3, :3] = R_0C.T; M[:3, 3] = -R_0C.T @ TC
    return M


class Pieza:
    """Prisma recto de altura h: 'disco' (diametro d), 'cuadrado' (lado d) o 'triangulo' (lado d), girado ang."""
    def __init__(self, forma, x, y, d, ang=0.0, h=H_PIEZA, albedo=0.80):
        self.forma, self.c, self.d, self.ang, self.h, self.albedo = forma, np.array([x, y]), d, ang, h, albedo
        if forma == 'cuadrado':
            v = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]])*d/2
        elif forma == 'triangulo':
            r = d/np.sqrt(3)
            v = np.array([[r*np.cos(a), r*np.sin(a)] for a in (np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3)])
        else:
            v = None
        if v is not None:
            ca, sa = np.cos(ang), np.sin(ang)
            self.v = v @ np.array([[ca, sa], [-sa, ca]]) + self.c

    def dentro(self, X, Y):
        if self.forma == 'disco':
            return (X - self.c[0])**2 + (Y - self.c[1])**2 <= (self.d/2)**2
        ok = np.ones(X.shape, bool)
        v = self.v
        for a, b in zip(v, np.roll(v, -1, axis=0)):      # poligono convexo, vertices en sentido antihorario
            ok &= (b[0] - a[0])*(Y - a[1]) - (b[1] - a[1])*(X - a[0]) >= 0
        return ok

    def area_px(self, f=F, Z=1.0):
        """Area aproximada de la cara de arriba en la imagen (a la distancia Z - h)."""
        e = f/(Z - self.h)
        if self.forma == 'disco':
            return np.pi*(self.d/2*e)**2
        if self.forma == 'cuadrado':
            return (self.d*e)**2
        return np.sqrt(3)/4*(self.d*e)**2


def vetas(X, Y):
    """La veta de la madera de la mesa: rayas onduladas a lo largo de y, cada 8 mm, de un 8 % de amplitud."""
    return 0.08*np.sin(2*np.pi*X/0.008 + 2.5*np.sin(2*np.pi*Y/0.15))


ESCENA = [Pieza('disco', 0.40, 0.20, 0.040),                 # la pieza: en P, el punto de la inversa de S10
          Pieza('cuadrado', 0.12, 0.34, 0.040, ang=0.35),     # dos piezas que no son la nuestra
          Pieza('triangulo', 0.56, 0.40, 0.050, ang=0.2)]


class Mesa:
    """La camara (K y distorsion de la unidad) a 1 m de la mesa, cenital: por defecto, en el plano del montaje,
    T_0C = Trans(0,3; 0,2; 1)·Rx(180°); con T_0C, donde de verdad la dejo su soporte (la unidad de S16)."""
    def __init__(self, K=K_CURSO, dist=DIST_CURSO, T_0C=None, ss=2):
        T = T_0C_NOMINAL if T_0C is None else np.asarray(T_0C, float)
        self.K, self.dist, self.ss = K, dist, ss
        self.R, self.o = T[:3, :3], T[:3, 3]                        # giro y origen de la camara en {0}
        us = (np.arange(W*ss) + 0.5)/ss - 0.5
        vs = (np.arange(H*ss) + 0.5)/ss - 0.5
        U, V = np.meshgrid(us, vs)
        pts = np.stack([U.ravel(), V.ravel()], -1).reshape(-1, 1, 2)
        n = cv2.undistortPointsIter(pts, K, dist, None, None,
                                    (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 60, 1e-12)).reshape(-1, 2)
        d = (self.R @ np.vstack([n.T, np.ones(len(n))])).T        # direccion del rayo en el mundo
        self.d = d
        self.X0, self.Y0 = self.corta(0.0)                         # donde corta cada rayo a la mesa

    def corta(self, z):
        lam = (z - self.o[2])/self.d[:, 2]
        return self.o[0] + lam*self.d[:, 0], self.o[1] + lam*self.d[:, 1]

    def foto(self, piezas=ESCENA, rng=None, luz=(1.0, 1.0), mesa=0.35, lateral=0.30, sombra=0.45,
             lampara=None, blur=0.7, ruido=2.0, textura=vetas, mascara=False):
        """luz = (en x = -0,1 m, en x = 0,7 m): la iluminacion de la mesa varia en linea recta a lo ancho.
        lampara: hacia donde esta la lampara, en metros de mesa por metro de altura (None: luz del techo, sin
        sombras; (1,73; 0,5), una lampara a 30° sobre la mesa, a la derecha). Devuelve la imagen
        en gris (uint8) y, si mascara, la verdad: un entero por pixel (0 mesa, i+1 la cara de arriba de la pieza i)."""
        rng = np.random.default_rng(0) if rng is None else rng
        X0, Y0 = self.X0, self.Y0
        L = luz[0] + (luz[1] - luz[0])*np.clip((X0 + 0.10)/0.80, 0, 1)
        alb = np.full(X0.shape, mesa)
        if textura is not None:
            alb = alb*(1 + textura(X0, Y0))
        verdad = np.zeros(X0.shape, np.int32)
        for i, p in enumerate(piezas):
            # solo los rayos que pasan cerca de la pieza (mas rapido: la escena entera son 1,2 millones de rayos)
            alc = p.d + (0 if lampara is None else np.hypot(*lampara)*p.h) + 0.01
            j = np.flatnonzero((X0 - p.c[0])**2 + (Y0 - p.c[1])**2 < alc**2)
            x0, y0 = X0[j], Y0[j]
            base = p.dentro(x0, y0)
            if lampara is not None:      # sombra: desde ese punto de la mesa, la lampara se ve a traves de la pieza
                somb = np.zeros(len(j), bool)
                for z in (0.25*p.h, 0.5*p.h, 0.75*p.h, p.h):
                    somb |= p.dentro(x0 + lampara[0]*z, y0 + lampara[1]*z)
                alb[j] = np.where(somb & ~base, alb[j]*sombra, alb[j])
            lam = (p.h - self.o[2])/self.d[j, 2]
            top = p.dentro(self.o[0] + lam*self.d[j, 0], self.o[1] + lam*self.d[j, 1])
            alb[j] = np.where(base & ~top, lateral, alb[j])     # el lateral: entra por el borde y no sale por la cara
            alb[j] = np.where(top, p.albedo, alb[j])
            verdad[j] = np.where(top, i + 1, verdad[j])
        img = 255*alb*L
        ss = self.ss
        img = img.reshape(H*ss, W*ss).reshape(H, ss, W, ss).mean(axis=(1, 3))
        img = cv2.GaussianBlur(img, (0, 0), blur)
        img = img + rng.normal(0, ruido, img.shape)
        img = np.clip(np.round(img), 0, 255).astype(np.uint8)
        if mascara:     # la verdad en la rejilla de pixeles: la pieza que cubre al menos 2 de los 4 subpixeles
            vs = verdad.reshape(H, ss, W, ss).transpose(0, 2, 1, 3).reshape(H, W, ss*ss)
            v = np.zeros((H, W), np.int32)
            for i in range(1, len(piezas) + 1):
                v[(vs == i).sum(-1)*2 >= ss*ss] = i
            return img, v
        return img


def a_la_mesa(uv, K, dist, T_C0, z=0.0):
    """Del pixel al plano z de {0}: quitar la distorsion, rayo K^-1·p, girarlo al mundo y cortarlo con el plano."""
    uv = np.atleast_2d(np.asarray(uv, float)).reshape(-1, 1, 2)
    n = cv2.undistortPointsIter(uv, K, np.asarray(dist, float), None, None,
                                (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-12)).reshape(-1, 2)
    n = np.column_stack([n, np.ones(len(n))])
    T = np.linalg.inv(np.asarray(T_C0, float))
    R, o = T[:3, :3], T[:3, 3]
    d = (R @ n.T).T
    lam = (z - o[2])/d[:, 2]
    return (o + lam[:, None]*d)[:, :2]


def ik2r(x, y, codo=+1):
    """La inversa cerrada del 2R de S10 (a1 = 0,35 m, a2 = 0,2 m); codo = +1, codo abajo."""
    c2 = (x*x + y*y - A1**2 - A2**2)/(2*A1*A2)
    q2 = codo*np.arccos(np.clip(c2, -1, 1))
    q1 = np.arctan2(y, x) - np.arctan2(A2*np.sin(q2), A1 + A2*np.cos(q2))
    return q1, q2


def piezas_en(mask, area_min=300, area_max=3000):
    """Componentes conexas de una mascara binaria: area, centroide (momentos), circularidad y caja."""
    n, et, st, cen = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for k in range(1, n):
        a = st[k, cv2.CC_STAT_AREA]
        if a < area_min or a > area_max:
            continue
        m = (et == k).astype(np.uint8)
        M = cv2.moments(m, binaryImage=True)
        cont = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0][0]
        per = cv2.arcLength(cont, True)
        x, y, w, h = st[k, :4]
        out.append(dict(area=M['m00'], u=M['m10']/M['m00'], v=M['m01']/M['m00'],
                        circ=4*np.pi*M['m00']/per**2, caja=(x, y, x + w, y + h)))
    return out


cam = Mesa()
LAMPARA = dict(luz=(0.5, 1.0), lampara=(1.73, 0.5))
I_t, V_t = cam.foto(mascara=True)
I_l, V_l = cam.foto(rng=np.random.default_rng(1), mascara=True, **LAMPARA)
print('Listo: la mesa, con la luz del techo y con la lámpara.')

## 1. Un núcleo aprendido de ejemplos se parece a Sobel

*En clase, sin detenerte: es la diapositiva 34.*

En el cuaderno S17a, los núcleos los elegía alguien: Sobel resta la columna izquierda de la derecha. Ahora, al revés: **no le decimos el núcleo, solo ejemplos**. Cada píxel de la foto, con su ruido, y sus 8 vecinos (9 números), y lo que debería salir: **la derivada en u de la misma foto sin ruido**, (I₀(u + 1) − I₀(u − 1))/2, que el simulador conoce. Son 307 200 ejemplos y 9 incógnitas. **Mínimos cuadrados** busca los 9 pesos que menos se equivocan: como la calibración de S16, que ajustaba los parámetros de la cámara para reducir el error de reproyección. Nadie le dice que promedie para quitar el ruido: lo descubre en los datos. Una red hace lo mismo con millones de pesos, a pequeños pasos que reducen el error, porque su problema no es lineal.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
SOBEL = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
In = I_t.astype(float)/255                                          # la foto, con el ruido del sensor
I0 = cam.foto(ruido=0.0).astype(float)/255                          # la misma foto sin ruido: solo un simulador la tiene
A9 = sliding_window_view(np.pad(In, 1, mode='edge'), (3, 3)).reshape(-1, 9)        # cada píxel con sus 8 vecinos
I0p = np.pad(I0, 1, mode='edge')
objetivo = ((I0p[1:-1, 2:] - I0p[1:-1, :-2])/2).ravel()            # lo que debería salir: la derivada en u, sin ruido
w9 = np.linalg.lstsq(A9, objetivo, rcond=None)[0].reshape(3, 3)
print(A9.shape[0], 'ejemplos y 9 incógnitas')
print('el núcleo aprendido, por 8:')
for fila in 8*w9:
    print('  ', '  '.join(f'{coma(np.round(x, 2) + 0.0):>6s}' for x in fila))
print('correlación con Sobel en u:', coma(np.corrcoef(w9.ravel(), SOBEL.ravel())[0, 1]))
for nom, k in (('la diferencia centrada, sin promediar', np.array([[0, 0, 0], [-0.5, 0, 0.5], [0, 0, 0]])), ('Sobel/8', SOBEL/8), ('el aprendido', w9)):
    o = cv2.filter2D(In, -1, k, borderType=cv2.BORDER_REPLICATE).ravel()
    print(f'{nom}: se aparta de la derivada sin ruido {coma(255*np.sqrt(np.mean((o - objetivo)**2)))} niveles por píxel')

## 2. Acertar al entrenar no es acertar

*En clase, sin detenerte: es la diapositiva 36. Tarda unos treinta segundos en Colab: fabrica 48 escenas.*

El clasificador más pequeño posible: **¿pieza o mesa?**, píxel a píxel, mirando su vecindario de 7 × 7 (49 números, de 0 a 1). Es una **regresión logística**: una sola neurona, con 49 pesos (un núcleo de 7 × 7 aprendido) y un umbral. Se **entrena** con unos píxeles y se **prueba** con otros, de escenas que nunca ha visto.

- **Sobreajuste:** con muy pocos ejemplos, el modelo acierta todos los suyos y falla con los nuevos: los ha memorizado. Se ve con la luz del techo y la media de 5 sorteos, mitad pieza y mitad mesa.
- La prueba se hace con **20 escenas nuevas por luz**: con pocas, el acierto cambia mucho de un sorteo a otro.
- **El dominio:** entrenado solo con la luz del techo, ¿qué pasa con la lámpara? ¿Y si se entrena con fotos de las dos luces?

In [ ]:
from sklearn.linear_model import LogisticRegression
K7 = 7

def escena_azar(rng, n=6):
    ps = []
    while len(ps) < n:
        f = rng.choice(['disco', 'cuadrado', 'triangulo']); x = rng.uniform(-0.05, 0.65); y = rng.uniform(-0.05, 0.45)
        if all(np.hypot(x - p.c[0], y - p.c[1]) > 0.07 for p in ps):
            ps.append(Pieza(f, x, y, rng.uniform(0.03, 0.05), ang=rng.uniform(0, np.pi)))
    return ps

def muestras(rng, n_esc, n_pix, luz='techo'):
    """n_pix píxeles por escena, mitad pieza y mitad mesa: cada uno, su vecindario de 7 x 7, y si es pieza."""
    X, y = [], []
    for e in range(n_esc):
        ps = escena_azar(rng)
        l = luz if luz != 'mezcla' else ('techo' if e % 2 == 0 else 'lampara')
        I_, V_ = cam.foto(ps, rng, mascara=True) if l == 'techo' else cam.foto(ps, rng, mascara=True, **LAMPARA)
        ventanas = sliding_window_view(np.pad(I_.astype(np.float32)/255, K7//2, mode='edge'), (K7, K7))
        es = V_ > 0
        pos, neg = np.argwhere(es), np.argwhere(~es)
        sel = np.vstack([pos[rng.choice(len(pos), n_pix//2)], neg[rng.choice(len(neg), n_pix//2)]])
        X.append(ventanas[sel[:, 0], sel[:, 1]].reshape(len(sel), -1)); y.append(es[sel[:, 0], sel[:, 1]])
    return np.vstack(X), np.concatenate(y)

t0 = time.time()
rng = np.random.default_rng(17)
X_tr, y_tr = muestras(rng, 4, 500, 'techo')        # entrenamiento: 4 escenas con la luz del techo
X_mz, y_mz = muestras(rng, 4, 500, 'mezcla')       # entrenamiento: 4 escenas, dos con cada luz
X_te, y_te = muestras(rng, 20, 100, 'techo')       # prueba: 20 escenas nuevas con la luz del techo
X_la, y_la = muestras(rng, 20, 100, 'lampara')     # prueba: 20 escenas nuevas con la lámpara
print(f'{len(X_tr)} píxeles por conjunto ({coma(time.time() - t0, 0)} s)')

m1 = LogisticRegression(max_iter=3000).fit(X_tr, y_tr)
print(f'Entrenada con el techo: entrenamiento {coma(100*m1.score(X_tr, y_tr), 1)} %; prueba, misma luz {coma(100*m1.score(X_te, y_te), 1)} %;'
      f' con la lámpara {coma(100*m1.score(X_la, y_la), 1)} %')
m2 = LogisticRegression(max_iter=3000).fit(X_mz, y_mz)
print(f'Entrenada con las dos luces: con el techo {coma(100*m2.score(X_te, y_te), 1)} %; con la lámpara {coma(100*m2.score(X_la, y_la), 1)} %')

rng_s = np.random.default_rng(170)
print('\nPocos ejemplos (con la luz del techo; media de 5 sorteos; prueba: las 20 escenas nuevas):')
for n in (6, 10, 20, 50, 200, 1000):
    acc = []
    for rep in range(5):
        pos_, neg_ = np.flatnonzero(y_tr), np.flatnonzero(~y_tr)
        idx = np.concatenate([rng_s.choice(pos_, n//2, replace=False), rng_s.choice(neg_, n - n//2, replace=False)])
        mm = LogisticRegression(max_iter=3000, C=1e4).fit(X_tr[idx], y_tr[idx])
        acc.append((mm.score(X_tr[idx], y_tr[idx]), mm.score(X_te, y_te)))
    acc = 100*np.array(acc)
    print(f'  {n:5d} ejemplos: entrenamiento {coma(acc[:, 0].mean(), 1)} %; prueba {coma(acc[:, 1].mean(), 1)} %')

## 3. El detector clásico del S17a, medido

*En clase, con el ejercicio de clase. Son las cuentas de las diapositivas 38 y 40.*

El detector del cuaderno S17a, convertido en **detector de discos**: umbral adaptativo (51 × 51, más 10), apertura de 3 × 3, componentes de 500 a 1200 píxeles y **circularidad mínima** (el umbral que se elige). Se mide en **20 escenas** con verdad conocida, la mitad con la lámpara: discos de 40 mm, y además cuadrados, triángulos y discos pequeños, de 22 mm, que no son la pieza. Con esas 20 se elige el umbral; después se comprueba en **40 escenas nuevas**: acertar con las escenas con las que se eligió no es acertar.

- Una caja detectada es un **acierto (VP, verdadero positivo)** si su **IoU** con la caja de un disco verdadero, la intersección entre la unión, es de 0,5 o más; si no, es una **caja falsa (FP)**. Cada disco sin caja es un **fallo (FN)**.
- **Precisión** = VP/(VP + FP): de lo que dice que es un disco, cuánto lo es. **Exhaustividad** = VP/(VP + FN): de los discos que hay, cuántos encuentra.

In [ ]:
def caja_de(V, k):
    ij = np.argwhere(V == k); return (ij[:, 1].min(), ij[:, 0].min(), ij[:, 1].max() + 1, ij[:, 0].max() + 1)

def iou(A, B):
    ix = max(0, min(A[2], B[2]) - max(A[0], B[0])); iy = max(0, min(A[3], B[3]) - max(A[1], B[1]))
    i = ix*iy
    return i/((A[2] - A[0])*(A[3] - A[1]) + (B[2] - B[0])*(B[3] - B[1]) - i)

print('IoU de (100; 100)-(180; 160) con (120; 110)-(200; 170):', coma(iou((100, 100, 180, 160), (120, 110, 200, 170)), 3))

def escena_det(rng):
    ps = []
    for f in ['disco']*rng.integers(1, 4) + list(rng.choice(['cuadrado', 'triangulo', 'disco_peq'], rng.integers(2, 5))):
        for _ in range(200):
            x, y = rng.uniform(-0.07, 0.67), rng.uniform(-0.07, 0.47)
            if all(np.hypot(x - p.c[0], y - p.c[1]) > 0.065 for p in ps):
                break
        if f == 'disco_peq':
            ps.append(Pieza('disco', x, y, 0.022))                    # un disco pequeño, de 22 mm: no es la pieza
        elif f == 'disco':
            ps.append(Pieza('disco', x, y, 0.040))
        else:
            ps.append(Pieza(f, x, y, rng.uniform(0.035, 0.05), ang=rng.uniform(0, np.pi)))
    return ps

def detector_clasico(I_):
    """Las cajas y circularidades de lo que pasa el área (500 a 1200 px), tras el umbral adaptativo y la apertura."""
    b = cv2.adaptiveThreshold(I_, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, -10)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    return [(p['caja'], p['circ']) for p in piezas_en(b, area_min=500, area_max=1200)]

def fabrica(semilla, n):
    """n escenas, la mitad con la lámpara: lo que da el detector (cajas y circularidades) y las cajas de los discos verdaderos."""
    rng_ = np.random.default_rng(semilla); esc = []
    for k in range(n):
        ps = escena_det(rng_)
        I_, V_ = cam.foto(ps, rng_, mascara=True, **(LAMPARA if k % 2 else {}))
        verdad = [caja_de(V_, i + 1) for i, p in enumerate(ps) if p.forma == 'disco' and p.d > 0.03 and (V_ == i + 1).sum() > 0]
        esc.append((detector_clasico(I_), verdad))
    return esc

t0 = time.time()
escenas = fabrica(19, 20)                  # para elegir el umbral
print(f'20 escenas, {sum(len(v) for _, v in escenas)} discos ({coma(time.time() - t0, 0)} s)')

def mide(circ_min, escenas=escenas):
    VP = FP = FN = 0
    for dets, verdad in escenas:
        usadas = set()
        for caja, circ in dets:
            if circ < circ_min:
                continue
            io = [(iou(caja, v), j) for j, v in enumerate(verdad) if j not in usadas]
            if io and max(io)[0] >= 0.5:
                usadas.add(max(io)[1]); VP += 1
            else:
                FP += 1
        FN += len(verdad) - len(usadas)
    return VP, FP, FN

for cm in (0.75, 0.94):
    VP, FP, FN = mide(cm)
    print(f'circularidad ≥ {coma(cm)}: VP {VP}, FP {FP}, FN {FN}; precisión {coma(VP/(VP + FP), 3)}; exhaustividad {coma(VP/(VP + FN), 3)}')

### Ejercicio de clase (en clase; se pone en común al acabar el taller)

Con 0,75 entran cuadrados; con 0,94 se pierden discos. **Elige con las 20 escenas una circularidad mínima que dé la precisión y la exhaustividad a 1, y mídela después en 40 escenas nuevas.** ¿Sigue dando 1 y 1? ¿Qué umbral de la banda elegirías, y por qué?

In [ ]:
# Ejercicio de clase: elegir con las 20 escenas y comprobar en 40 nuevas
t0 = time.time()
prueba = fabrica(2020, 40)                 # 40 escenas nuevas, para comprobar
print(f'40 escenas nuevas, {sum(len(v) for _, v in prueba)} discos ({coma(time.time() - t0, 0)} s)')
for cm in (0.80, 0.85, 0.88, 0.90, 0.92):
    r = []
    for esc in (escenas, prueba):
        VP, FP, FN = mide(cm, esc)
        r.append(f'precisión {coma(VP/(VP + FP) if VP + FP else 1, 3)}, exhaustividad {coma(VP/(VP + FN), 3)}')
    print(f'circularidad ≥ {coma(cm)}: en las 20, {r[0]} ; en las 40 nuevas, {r[1]}')

cms = np.round(np.arange(0.60, 0.981, 0.01), 2)
plt.figure(figsize=(7, 3.6))
for esc, ls, t in ((escenas, '-', '20 escenas'), (prueba, '--', '40 nuevas')):
    P_, R_ = [], []
    for cm in cms:
        VP, FP, FN = mide(cm, esc); P_.append(VP/(VP + FP) if VP + FP else 1.0); R_.append(VP/(VP + FN))
    plt.plot(cms, P_, ls, color=IQS_AZUL, lw=2, label=f'precisión, {t}'); plt.plot(cms, R_, ls, color=AMBAR, lw=2, label=f'exhaustividad, {t}')
plt.xlabel('circularidad mínima'); plt.ylim(0.5, 1.03); plt.grid(True); plt.legend(fontsize=8); ejes(plt.gca()); plt.show()

## 4. Un detector aprendido: NanoDet-Plus

*En clase. Son las diapositivas 35 y 37.*

NanoDet-Plus es una red convolucional de **una etapa**: mira la imagen una vez (de 416 × 416) y da todas las cajas, como el camino de Redmon et al. (2016). Está entrenada con COCO, un conjunto de fotos con **80 clases** (personas, animales, objetos de casa…). OpenCV la ejecuta en la CPU con su módulo `dnn`, sin GPU.

La celda de abajo prepara la foto (la escala a 416 × 416 y la normaliza), ejecuta la red y **decodifica** su salida: para cada posición, una confianza por clase y la distancia a los cuatro lados de la caja. Ejecútala sin detenerte: el código está para quien quiera mirarlo. Después, la **supresión de no máximos** (`cv2.dnn.NMSBoxes`) quita las cajas repetidas: de las que se solapan (IoU > 0,6), se queda la más confiada.

In [ ]:
COCO = ('persona,bicicleta,coche,moto,avión,autobús,tren,camión,barco,semáforo,boca de incendios,señal de stop,parquímetro,'
        'banco,pájaro,gato,perro,caballo,oveja,vaca,elefante,oso,cebra,jirafa,mochila,paraguas,bolso,corbata,maleta,'
        'disco volador,esquís,tabla de nieve,pelota,cometa,bate,guante,monopatín,tabla de surf,raqueta,botella,copa,taza,'
        'tenedor,cuchillo,cuchara,cuenco,plátano,manzana,bocadillo,naranja,brócoli,zanahoria,perrito caliente,pizza,rosquilla,'
        'tarta,silla,sofá,planta,cama,mesa,inodoro,televisor,portátil,ratón,mando a distancia,teclado,teléfono móvil,'
        'microondas,horno,tostadora,fregadero,nevera,libro,reloj,jarrón,tijeras,osito de peluche,secador,cepillo de dientes').split(',')
PASOS, RM, LADO = (8, 16, 32, 64), 7, 416
anclas = []
for st in PASOS:
    n = LADO//st; xv, yv = np.meshgrid(np.arange(n)*st, np.arange(n)*st)
    anclas.append(np.column_stack((xv.ravel() + 0.5*(st - 1), yv.ravel() + 0.5*(st - 1))))
MEDIA_ = np.array([103.53, 116.28, 123.675], np.float32); DESV_ = np.array([57.375, 57.12, 58.395], np.float32)
net = cv2.dnn.readNet(RUTA_DETECTOR) if RUTA_DETECTOR else None

def detectar(rgb, umbral=0.35, solape=0.6):
    """Cajas (x0, y0, x1, y1), clases y confianzas de NanoDet sobre una foto en color (RGB)."""
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR); h, w = bgr.shape[:2]; e = LADO/max(h, w)
    lienzo = np.zeros((LADO, LADO, 3), np.uint8); lienzo[:int(h*e), :int(w*e)] = cv2.resize(bgr, (int(w*e), int(h*e)))
    net.setInput(cv2.dnn.blobFromImage((lienzo.astype(np.float32) - MEDIA_)/DESV_))
    salidas = net.forward(net.getUnconnectedOutLayersNames())
    # se emparejan por su forma (80 clases o 32 números de caja), no por su orden: así vale en OpenCV 4 y 5
    clases = sorted([o for o in salidas if o.shape[-1] == 80], key=lambda o: -o.size)
    cajas = sorted([o for o in salidas if o.shape[-1] == 4*(RM + 1)], key=lambda o: -o.size)
    B, P = [], []
    for st, cs, bp, a in zip(PASOS, clases, cajas, anclas):
        cs, bp = cs.reshape(-1, 80), bp.reshape(-1, 4*(RM + 1))
        x = np.exp(bp.reshape(-1, RM + 1)); d = (x/x.sum(1, keepdims=True)) @ np.arange(RM + 1); d = d.reshape(-1, 4)*st
        B.append(np.column_stack([a[:, 0] - d[:, 0], a[:, 1] - d[:, 1], a[:, 0] + d[:, 2], a[:, 1] + d[:, 3]])); P.append(cs)
    B, P = np.concatenate(B)/e, np.concatenate(P)
    cl, cf = P.argmax(1), P.max(1)
    xywh = B.copy(); xywh[:, 2:] -= xywh[:, :2]
    quedan = cv2.dnn.NMSBoxes(xywh.tolist(), cf.tolist(), umbral, solape)
    return [(COCO[int(cl[i])], float(cf[i]), B[i]) for i in np.array(quedan).ravel()], int((cf >= umbral).sum())

RESPALDO = {'el café (umbral 0,35)': [('taza', 0.59), ('mesa', 0.54), ('mesa', 0.35)], 'el gato (umbral 0,35)': [('gato', 0.57)],
            'la mesa (umbral 0,2)': [('pájaro', 0.40), ('disco volador', 0.33), ('teléfono móvil', 0.22)]}
if net is None:
    print('Sin el detector, lo que habría salido (medido con OpenCV 4.14.0; aquí no hay red):')
    for nom, dets in RESPALDO.items():
        print(f'  {nom}: ' + '; '.join(f'{cl} {coma(cf)}' for cl, cf in dets))
else:
    from skimage import data
    fotos = {'el café': data.coffee(), 'el gato': data.chelsea(), 'la mesa': cv2.cvtColor(I_t, cv2.COLOR_GRAY2RGB)}
    fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
    for ax, (nom, im) in zip(axs, fotos.items()):
        umbral = 0.2 if nom == 'la mesa' else 0.35
        t = []
        for _ in range(5):
            t0 = time.time(); dets, antes = detectar(im, umbral); t.append(time.time() - t0)
        print(f'{nom}: {coma(1000*np.median(t), 0)} ms; {antes} cajas por encima de {coma(umbral, 2)} y {len(dets)} tras suprimir las repetidas:')
        ax.imshow(im); ax.set_title(nom); sin_ejes(ax)
        for cl, cf, b in dets:
            print(f'   {cl} {coma(cf)}  [{b[0]:.0f}, {b[1]:.0f}, {b[2]:.0f}, {b[3]:.0f}]')
            ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=IQS_VERDE, lw=2))
            ax.text(b[0], b[1], f'{cl} {coma(cf)}', color='white', fontsize=9, va='bottom', bbox=dict(fc=IQS_VERDE, ec='none', pad=1))
    plt.tight_layout(); plt.show()

In [ ]:
if net is not None:
    capas = net.getLayerNames()
    print('NanoDet:', len(capas), 'capas; las tres primeras:', ', '.join(net.getLayer(c).type for c in capas[:3]))
    net.setInput(np.random.default_rng(0).normal(0, 1, (1, 3, 416, 416)).astype(np.float32))
    o_c, o_r, o_p = net.forward(list(capas[:3]))          # lo que sale de cada una con una entrada al azar
    neg = o_c < 0
    print(f'  la convolución: de 416 x 416 a {o_c.shape[-1]} x {o_c.shape[-2]} (los núcleos se aplican cada 2 píxeles);'
          f' la «ReLU» multiplica lo negativo por {coma(float(np.median(o_r[neg]/o_c[neg])))} (una ReLU con fuga);'
          f' la reducción deja {o_p.shape[-1]} x {o_p.shape[-2]}')
    w1 = net.getParam(capas[0], 0)                 # la primera capa: 24 núcleos de 3 x 3 x 3 (canal, alto, ancho)
    print('la primera capa:', w1.shape, '=', w1.size, 'pesos')
    gris = w1.sum(1)                               # cada núcleo, visto en gris: la suma de sus tres canales
    def parecido(a, b):
        a, b = a.ravel() - a.mean(), b.ravel() - b.mean(); return float(a @ b/np.linalg.norm(a)/np.linalg.norm(b))
    fig, axs = plt.subplots(3, 8, figsize=(12, 4.6))
    for k, ax in enumerate(axs.ravel()):
        su, sv = parecido(gris[k], SOBEL), parecido(gris[k], SOBEL.T)
        lim = np.abs(gris[k]).max(); ax.imshow(gris[k], cmap='RdBu_r', vmin=-lim, vmax=lim); sin_ejes(ax)
        ax.set_title(f'{k}: {coma(max(abs(su), abs(sv)))}', fontsize=9, color=IQS_VERDE if max(abs(su), abs(sv)) > 0.85 else 'black')
    plt.suptitle('Los 24 primeros núcleos de NanoDet, en gris, y su parecido con Sobel (en u o en v)'); plt.tight_layout(); plt.show()
    print('el núcleo 11, con Sobel en u:', coma(parecido(gris[11], SOBEL)))

### Ejercicio 1 (en casa): lo que ha aprendido la neurona

La neurona de la sección 2, entrenada con la luz del techo (`m1`), tiene 49 pesos: `m1.coef_.reshape(7, 7)`. Dibújalos con `plt.imshow`, y también los de la entrenada con las dos luces (`m2`). ¿Qué núcleo ha aprendido cada una? ¿Por qué la primera falla con la lámpara y la segunda no tanto?

In [ ]:
# Ejercicio 1: prueba aquí
# fig, axs = plt.subplots(1, 2, figsize=(8, 3.6))
# for ax, (m, t) in zip(axs, ((m1, 'con el techo'), (m2, 'con las dos luces'))):
#     w = m.coef_.reshape(7, 7); lim = np.abs(w).max()
#     ax.imshow(w, cmap='RdBu_r', vmin=-lim, vmax=lim); ax.set_title(t)
#     print(t, ': suma de los pesos', coma(w.sum()), '; el del centro', coma(w[3, 3]), '; umbral', coma(m.intercept_[0]))
# plt.show()

### Ejercicio 2 (en casa): un detector de cuadrados

Cambia el detector de la sección 3 para que busque **cuadrados** (circularidad entre 0,70 y 0,80 y área de 500 a 1500 píxeles). Mide su precisión y su exhaustividad en las mismas 20 escenas. ¿Es tan fácil como el de discos? ¿Por qué?

In [ ]:
# Ejercicio 2: prueba aquí
# rng19 = np.random.default_rng(19); VP = FP = FN = 0
# for k in range(20):
#     ps = escena_det(rng19)
#     I_, V_ = cam.foto(ps, rng19, mascara=True, **(LAMPARA if k % 2 else {}))
#     verdad = [caja_de(V_, i + 1) for i, p in enumerate(ps) if p.forma == 'cuadrado' and (V_ == i + 1).sum() > 0]
#     b = cv2.morphologyEx(cv2.adaptiveThreshold(I_, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, -10), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
#     dets = [p['caja'] for p in piezas_en(b, 500, 1500) if 0.70 <= p['circ'] <= 0.80]     # prueba también hasta 0,86
#     usadas = set()
#     for c_ in dets:
#         io = [(iou(c_, v), j) for j, v in enumerate(verdad) if j not in usadas]
#         if io and max(io)[0] >= 0.5: usadas.add(max(io)[1]); VP += 1
#         else: FP += 1
#     FN += len(verdad) - len(usadas)
# print('VP', VP, 'FP', FP, 'FN', FN, '; precisión', coma(VP/max(VP + FP, 1), 3), '; exhaustividad', coma(VP/max(VP + FN, 1), 3))

### Ejercicio 3 (en casa): la caja de la red, con la lámpara

Con la lámpara (`I_l`) y el umbral de confianza en 0,2, ¿qué cajas da NanoDet sobre la mesa? Calcula el IoU de la caja del «disco volador» con la caja verdadera del disco (`caja_de(V_l, 1)`), y compáralo con el de la luz del techo (`caja_de(V_t, 1)`). ¿Por qué baja? ¿Qué le pasaría a la exhaustividad si se exigiera un IoU de 0,7?

In [ ]:
# Ejercicio 3: prueba aquí
# for nom, I_, V_ in (('techo', I_t, V_t), ('lámpara', I_l, V_l)):
#     dets, _ = detectar(cv2.cvtColor(I_, cv2.COLOR_GRAY2RGB), 0.2)
#     for cl, cf, b in dets:
#         if cl == 'disco volador':
#             bi = [int(round(x)) for x in b]                  # la caja, en píxeles enteros
#             print(nom, coma(cf), bi, 'IoU', coma(iou(bi, caja_de(V_, 1))))

---

## Soluciones

**Ejercicio de clase.** Con 0,75 entran los cuadrados (circularidad 0,73 a 0,84): precisión 0,678, exhaustividad 1. Con 0,94 se pierden los ocho discos que no llegan a 0,94 (el que menos, 0,92): precisión 1, exhaustividad 0,8. Con las 20 escenas, cualquier umbral entre 0,85 y 0,92 da las dos a 1. En las 40 nuevas, 0,85 ya deja pasar una caja falsa (precisión 0,989: algo que no es un disco mide 0,86), y de 0,88 a 0,92 siguen en 1 y 1. Lo sensato es 0,90, en el centro de la banda, lejos de sus dos bordes: un umbral elegido en el borde de la banda funciona con las escenas con las que se eligió y falla con las nuevas. Es «acertar al entrenar no es acertar», con un detector clásico de un solo parámetro. Con luz controlada y piezas conocidas, el clásico no falla; el umbral mueve las dos medidas en sentidos opuestos, y por eso un detector se da con las dos y con su umbral.

**Ejercicio 1.** La entrenada con la luz del techo aprende un suavizado y un umbral: sus 49 pesos son positivos, el mayor en el centro (1,72), y pasado a gris dice «pieza si la media ponderada del vecindario pasa de 124». La entrenada con las dos luces concentra más el peso en el centro (3,17), pone pesos negativos en la fila de arriba y en una esquina (hasta −0,39) y baja el umbral a 103: empieza, apenas, a comparar el centro con su entorno, como el umbral adaptativo, y por eso con la lámpara acierta más (96,0 % frente a 83,7 %). Con un vecindario de 7 × 7, más pequeño que una pieza (32 px), no puede ir mucho más lejos: haría falta un vecindario mayor o más capas. Es el campo receptivo de la diapositiva 35.

**Ejercicio 2.** Con circularidad de 0,70 a 0,80: precisión 1 y exhaustividad 0,667 (16 de 24 cuadrados; ninguna caja falsa). Los cuadrados miden de 0,73 a 0,84, según su tamaño y su giro: un borde en diagonal es una escalera de píxeles y alarga el perímetro. Con 0,70 a 0,86 salen los 24 y ninguna caja falsa (los triángulos no pasan de 0,67 y los discos empiezan en 0,92). Funciona, pero con poco margen: la circularidad separa bien un disco de lo demás, y peor un cuadrado de un disco deformado o de un triángulo romo. Para cuadrados, mejor contar sus esquinas (sección 4 del S17a) o ajustar un polígono al contorno.

**Ejercicio 3.** Con la luz del techo, la caja del «disco volador» (0,33) tiene un IoU de 0,91 con la verdadera; con la lámpara (0,23), 0,59: la caja se estira hacia la izquierda para abarcar la sombra, que para la red es parte del objeto. Con un IoU mínimo de 0,7, esa detección contaría como una caja falsa y un disco perdido: bajaría la exhaustividad (y la precisión). La red no sabe qué es nuestra pieza (la llama disco volador, una de sus 80 clases) ni que la sombra no es suya: habría que entrenarla con nuestras fotos, con las dos luces.

---

## Para llevarse de esta sesión

- **La operación es la misma**; lo que cambia es de dónde salen los pesos: un núcleo ajustado con ejemplos sale casi Sobel (y aprende solo a promediar el ruido), y la primera capa de una red tiene núcleos que se le parecen.
- **Acertar al entrenar no es acertar**: con pocos ejemplos, el modelo los memoriza; se mide con datos que no ha visto y de su dominio. Lo que no ha visto (otra luz), no lo sabe.
- **Un detector se mide**: IoU para saber si la caja está bien puesta; precisión y exhaustividad, con su umbral, que las mueve al revés.
- **Una red preentrenada no conoce nuestras piezas**: el disco es un «disco volador». Y tarda: su latencia se mide en el ordenador donde va a correr.
- **La geometría no se aprende**: para pasar de la caja a milímetros sigue haciendo falta la K de S16.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*